# 02.3 — Build agents lab

1. The object model, made concrete: agent, thread, message, run, run step
2. Function calling the hard way — handle `requires_action` yourself
3. Function calling the easy way — `enable_auto_function_calls`
4. Conversation memory: what a thread actually costs you
5. Code Interpreter
6. File Search (managed vector store) vs Azure AI Search (your index)
7. OpenAPI spec tool and Bing grounding — the shapes, and when each is right
8. Run steps as an audit trail
9. Delete every agent, thread, file, and vector store

**Cost:** tokens, plus Code Interpreter session time and File Search vector
storage. Both are stopped by the cleanup cell. Run cleanup even if a cell fails.

**Prerequisites:** `gpt-4o-mini` deployed; `azure-ai-projects` and
`azure-ai-agents` installed. Section 6b additionally needs an Azure AI Search
connection on the project — it degrades gracefully if absent.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))
from ai103 import cfg, credential, chat_client, project_client

project = project_client()
agents = project.agents          # the Agents client hangs off AIProjectClient

# Track everything we create so cleanup is exhaustive.
CREATED = {"agents": [], "threads": [], "files": [], "vector_stores": []}

print("project :", cfg["AZURE_AI_PROJECT_ENDPOINT"])
print("model   :", cfg["MODEL_MINI"])

## 1. The object model

```
Agent    model + instructions + tools. Persistent. Stateless by itself.
Thread   a conversation. Persistent. Owned by NO agent.
Message  one turn in a thread. Immutable.
Run      one execution of ONE agent against ONE thread. Has a status.
RunStep  what the run did: message_creation or tool_calls. Your audit trail.
```

Start with the simplest possible agent and watch each object appear.

In [ ]:
basic = agents.create_agent(
    model=cfg["MODEL_MINI"],
    name="ai103-basic",
    instructions="You are terse. Answer in one sentence.",
)
CREATED["agents"].append(basic.id)
print("agent   :", basic.id, basic.name)

thread = agents.threads.create()
CREATED["threads"].append(thread.id)
print("thread  :", thread.id)

agents.messages.create(thread_id=thread.id, role="user",
                       content="What is a run step?")

run = agents.runs.create_and_process(thread_id=thread.id, agent_id=basic.id)
print("run     :", run.id, "->", run.status)
print("usage   :", run.usage)

In [ ]:
from azure.ai.agents.models import ListSortOrder


def print_thread(thread_id):
    for m in agents.messages.list(thread_id=thread_id, order=ListSortOrder.ASCENDING):
        text = "".join(p.text.value for p in m.content if p.type == "text")
        print(f"[{m.role}] {text}")


print_thread(thread.id)

In [ ]:
# A thread is not owned by an agent. Prove it: run a DIFFERENT agent on the SAME
# thread. This primitive is what handoff orchestration is built on (unit 02.4).
pirate = agents.create_agent(
    model=cfg["MODEL_MINI"],
    name="ai103-second-voice",
    instructions="You are a 17th-century ship's quartermaster. Stay in character.",
)
CREATED["agents"].append(pirate.id)

agents.messages.create(thread_id=thread.id, role="user",
                       content="Say that again, but in your own voice.")
agents.runs.create_and_process(thread_id=thread.id, agent_id=pirate.id)

print_thread(thread.id)

> **Exam note.** One thread, two agents, shared history. Threads are the memory;
> agents are interchangeable readers of it. That is why "conversation memory" in
> an agent solution means *the thread*, and why you never resend history yourself.

## 2. Function calling — the manual loop

The model **never runs your code**. It emits a tool call; your application executes
it and submits the result. Do it by hand once, because everything in unit 02.4
(approvals, audit, rate limiting) hooks into exactly this seam.

Note how much work the `description` fields are doing — the model routes entirely on
those strings.

In [ ]:
import json

ORDERS = {
    "ORD-12345": {"status": "in_transit", "carrier": "Northwind", "eta": "2026-09-09", "total_usd": 412.00},
    "ORD-99887": {"status": "delivered", "carrier": "Northwind", "eta": "2026-08-30", "total_usd": 89.50},
}


def get_order_status(order_id: str) -> str:
    """Look up the current status of an order."""
    print(f"    >> get_order_status({order_id!r}) executing locally")
    order = ORDERS.get(order_id.upper())
    if not order:
        return json.dumps({"error": "not_found", "order_id": order_id})
    return json.dumps({"order_id": order_id.upper(), **order})


def open_refund_case(order_id: str, reason: str) -> str:
    """Open a refund case. Deliberately requires a human to approve later."""
    print(f"    >> open_refund_case({order_id!r}, {reason!r}) executing locally")
    return json.dumps({"case_id": "CASE-7781", "order_id": order_id.upper(),
                       "state": "awaiting_human_approval", "reason": reason})


LOCAL_FUNCTIONS = {"get_order_status": get_order_status, "open_refund_case": open_refund_case}

# The schema IS the routing logic. Descriptions say WHEN to call, not just what.
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": (
                "Look up the current shipping status, carrier, ETA and total for an "
                "order. Call this whenever the user supplies an order ID. Never guess "
                "an order ID; ask for one instead."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "Order ID in the form ORD-12345"}
                },
                "required": ["order_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "open_refund_case",
            "description": (
                "Open a refund case for human review. Call this only after confirming "
                "the order exists. This does NOT issue a refund."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "Order ID in the form ORD-12345"},
                    "reason": {
                        "type": "string",
                        "enum": ["damaged", "late", "wrong_item", "changed_mind"],
                        "description": "Why the customer wants a refund",
                    },
                },
                "required": ["order_id", "reason"],
            },
        },
    },
]
print(f"{len(TOOL_SCHEMAS)} tool schemas defined")

In [ ]:
SUPPORT_INSTRUCTIONS = """You are Contoso's support triage assistant.

- Use get_order_status whenever the user supplies an order ID. Never invent an ID.
- You may open a refund case. You may NEVER promise a refund, a credit, or a date.
  Refunds are approved by a human.
- If a tool returns an error, tell the user plainly. Do not retry more than once.
- If the request is outside Contoso support, decline in one sentence."""

manual_agent = agents.create_agent(
    model=cfg["MODEL_MINI"],
    name="ai103-support-manual",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=TOOL_SCHEMAS,     # raw schemas — no auto-execution registered
)
CREATED["agents"].append(manual_agent.id)
print("created", manual_agent.id)

In [ ]:
import time


def run_manually(thread_id, agent_id, verbose=True):
    """Create a run and drive the requires_action loop by hand.

    This is the seam where approval gates, audit logging, and rate limiting live.
    """
    run = agents.runs.create(thread_id=thread_id, agent_id=agent_id)

    while run.status in ("queued", "in_progress", "requires_action"):
        if run.status == "requires_action":
            calls = run.required_action.submit_tool_outputs.tool_calls
            outputs = []
            for call in calls:
                name = call.function.name
                args = json.loads(call.function.arguments or "{}")
                if verbose:
                    print(f"  requires_action -> {name}({args})")
                fn = LOCAL_FUNCTIONS.get(name)
                result = fn(**args) if fn else json.dumps({"error": f"unknown tool {name}"})
                # EVERY tool_call_id must be answered in ONE submit call.
                outputs.append({"tool_call_id": call.id, "output": result})

            run = agents.runs.submit_tool_outputs(
                thread_id=thread_id, run_id=run.id, tool_outputs=outputs
            )
            continue

        time.sleep(0.6)
        run = agents.runs.get(thread_id=thread_id, run_id=run.id)

    if verbose:
        print(f"  final status: {run.status}")
        if run.status == "failed":
            print("  last_error:", run.last_error)
    return run


t2 = agents.threads.create()
CREATED["threads"].append(t2.id)
agents.messages.create(
    thread_id=t2.id, role="user",
    content="Order ORD-12345 turned up smashed. Where is it and can I get my money back?",
)

r = run_manually(t2.id, manual_agent.id)
print()
print_thread(t2.id)

Three things to notice:

- The `>>` lines prove the function ran **in this Python process**, not in Azure.
- The run may pause for `requires_action` more than once — the model can chain
  `get_order_status` then `open_refund_case` after reading the first result.
- The assistant does not promise a refund, because the instructions forbade it.
  Authority limits belong in instructions, and unit 02.4 shows why instructions
  alone are not a sufficient control.

> **Exam note.** If you never call `submit_tool_outputs`, the run stays in
> `requires_action` until it hits the run window and becomes `expired`. "The agent
> called the function but nothing happened" is always this.

## 3. Function calling — the automatic loop

`FunctionTool` derives the JSON schema from your Python signature and docstring;
`enable_auto_function_calls` registers the callables so `create_and_process`
executes them for you. Less code — and no seam to hang an approval on.

In [ ]:
from azure.ai.agents.models import FunctionTool, ToolSet

functions = FunctionTool({get_order_status, open_refund_case})
toolset = ToolSet()
toolset.add(functions)

agents.enable_auto_function_calls(toolset)   # the SDK will now execute tool calls

auto_agent = agents.create_agent(
    model=cfg["MODEL_MINI"],
    name="ai103-support-auto",
    instructions=SUPPORT_INSTRUCTIONS,
    toolset=toolset,
)
CREATED["agents"].append(auto_agent.id)

t3 = agents.threads.create()
CREATED["threads"].append(t3.id)
agents.messages.create(thread_id=t3.id, role="user", content="Status of ORD-99887?")

run3 = agents.runs.create_and_process(thread_id=t3.id, agent_id=auto_agent.id)
print("status:", run3.status)
print()
print_thread(t3.id)

| | manual `create` + poll | `create_and_process` + auto functions |
|---|---|---|
| Lines of code | ~20 | ~2 |
| You see `requires_action` | yes | no |
| Can require human approval before a tool runs | **yes** | no |
| Can audit/authorise every call | **yes** | only after the fact, via run steps |
| Right for | anything with side effects | read-only helpers |

> **Exam note.** "Approval before tool execution" or "audit every invocation"
> always points at the manual loop (or the MCP tool's built-in approval flow).

## 4. Conversation memory costs money

The thread is the memory, and every run replays the whole thread. Watch
`prompt_tokens` climb across turns you never resent.

In [ ]:
t4 = agents.threads.create()
CREATED["threads"].append(t4.id)

turns = [
    "Hi, I have a problem with an order.",
    "It's ORD-12345.",
    "It arrived damaged.",
    "So what happens next?",
]

print(f"{'turn':<6}{'prompt':>9}{'completion':>12}{'total':>9}")
cumulative = 0
for i, text in enumerate(turns, start=1):
    agents.messages.create(thread_id=t4.id, role="user", content=text)
    run = agents.runs.create_and_process(thread_id=t4.id, agent_id=auto_agent.id)
    u = run.usage
    cumulative += u.total_tokens
    print(f"{i:<6}{u.prompt_tokens:>9}{u.completion_tokens:>12}{u.total_tokens:>9}")

print(f"\ncumulative tokens for a 4-turn conversation: {cumulative}")

Notice turn 3 — the user typed "It arrived damaged", three words, and the prompt was
hundreds of tokens. You paid to replay instructions, tool schemas, and every prior
message. In a 40-turn support conversation this dominates the bill.

Mitigations, in the order you should reach for them:

1. Shorter instructions and fewer tools per agent (both are replayed every run).
2. A new thread per session rather than per customer.
3. Summarise on close and store the summary in your own database; seed the next
   thread with the summary rather than the transcript.
4. `truncation_strategy` on the run to cap how many recent messages are replayed.

Threads also persist user content until deleted — a **data-retention** decision, not
just a cost one.

## 5. Code Interpreter

Runs Python in a Foundry-managed sandbox. Use it for maths, data analysis, and chart
generation — anything where the model writing and *executing* code beats the model
guessing an answer.

> **Cost.** Code Interpreter is billed per session on top of tokens. A session
> stays warm for up to an hour with a 30-minute idle timeout, and concurrent
> conversations create separate sessions.

In [ ]:
import os, tempfile
from azure.ai.agents.models import CodeInterpreterTool, FilePurpose

csv_path = os.path.join(tempfile.gettempdir(), "ai103_tickets.csv")
with open(csv_path, "w", encoding="utf-8") as fh:
    fh.write("month,severity,tickets\n")
    for month, s1, s2, s3 in [
        ("Jan", 4, 22, 118), ("Feb", 2, 31, 140), ("Mar", 9, 18, 96),
        ("Apr", 3, 25, 155), ("May", 7, 40, 132), ("Jun", 1, 15, 101),
    ]:
        fh.write(f"{month},S1,{s1}\n{month},S2,{s2}\n{month},S3,{s3}\n")

uploaded = agents.files.upload_and_poll(file_path=csv_path, purpose=FilePurpose.AGENTS)
CREATED["files"].append(uploaded.id)
print("uploaded file:", uploaded.id)

ci = CodeInterpreterTool(file_ids=[uploaded.id])

analyst = agents.create_agent(
    model=cfg["MODEL_MINI"],
    name="ai103-analyst",
    instructions=(
        "You are a data analyst. Use the code interpreter to compute answers from the "
        "attached data. Never estimate a number you could calculate. State the figures."
    ),
    tools=ci.definitions,
    tool_resources=ci.resources,   # tools AND tool_resources are both required
)
CREATED["agents"].append(analyst.id)

t5 = agents.threads.create()
CREATED["threads"].append(t5.id)
agents.messages.create(
    thread_id=t5.id, role="user",
    content="Which month had the worst S1:S3 ratio, and what was the ratio? Show the numbers.",
)
run5 = agents.runs.create_and_process(thread_id=t5.id, agent_id=analyst.id)
print("status:", run5.status)
print()
print_thread(t5.id)

In [ ]:
# The run steps show the code the sandbox actually executed — the audit trail for
# a Code Interpreter answer.
for step in agents.run_steps.list(thread_id=t5.id, run_id=run5.id):
    details = step.step_details
    if getattr(details, "type", "") == "tool_calls":
        for call in details.tool_calls:
            ci_call = getattr(call, "code_interpreter", None)
            if ci_call is not None:
                print("--- executed code ---")
                print(ci_call.input[:900])

## 6a. File Search — Foundry's managed vector store

Foundry chunks, embeds, and stores the files for you. Fast to stand up; you control
nothing about the retrieval, and the content is duplicated into the agent service.

> **Cost.** Vector stores are billed for stored data per day. The cleanup cell
> deletes it.

In [ ]:
from azure.ai.agents.models import FileSearchTool

policy_path = os.path.join(tempfile.gettempdir(), "ai103_policy.md")
with open(policy_path, "w", encoding="utf-8") as fh:
    fh.write(
        "# Contoso Returns Policy\n\n"
        "Unopened items may be returned within 30 days for a full refund.\n"
        "Opened items receive store credit only, valid 12 months.\n"
        "Refunds are issued within 5 business days of warehouse receipt.\n\n"
        "# CX-4400 Warranty\n\n"
        "36 months from shipment. Void outside -10C to 55C.\n"
        "Claims must be filed within 14 days of fault discovery.\n"
    )

policy_file = agents.files.upload_and_poll(file_path=policy_path, purpose=FilePurpose.AGENTS)
CREATED["files"].append(policy_file.id)

store = agents.vector_stores.create_and_poll(file_ids=[policy_file.id], name="ai103-policy-store")
CREATED["vector_stores"].append(store.id)
print("vector store:", store.id, store.status)

fs = FileSearchTool(vector_store_ids=[store.id])

librarian = agents.create_agent(
    model=cfg["MODEL_MINI"],
    name="ai103-librarian",
    instructions=(
        "Answer only from the attached policy documents. Cite the document. "
        "If the answer is not in them, say so."
    ),
    tools=fs.definitions,
    tool_resources=fs.resources,
)
CREATED["agents"].append(librarian.id)

t6 = agents.threads.create()
CREATED["threads"].append(t6.id)
agents.messages.create(thread_id=t6.id, role="user",
                       content="I opened it. Refund or credit, and how long do I have for a warranty claim?")
run6 = agents.runs.create_and_process(thread_id=t6.id, agent_id=librarian.id)
print("status:", run6.status)
print()
print_thread(t6.id)

In [ ]:
# Citations arrive as ANNOTATIONS on the assistant message, not as text you parse.
for m in agents.messages.list(thread_id=t6.id, order=ListSortOrder.DESCENDING):
    if m.role == "assistant":
        for part in m.content:
            for ann in getattr(part.text, "annotations", []) or []:
                print("annotation:", getattr(ann, "type", "?"), "|", getattr(ann, "text", ""))
        break

## 6b. Azure AI Search tool — your index, your rules

The enterprise answer. The agent queries an index **you** built, with your chunking,
your filters, and your hybrid + semantic configuration — and other applications can
use the same index.

It needs a project **connection** to the Search service. If your project has none,
this cell reports that and moves on.

| | File Search | Azure AI Search tool |
|---|---|---|
| Who chunks | Foundry | you |
| Retrieval modes | managed vector search | keyword / vector / hybrid / semantic |
| Filters, scoring profiles | no | yes |
| Data duplicated into agent service | yes | no |
| Shared with other apps | no | yes |
| Right for | ad-hoc files, prototypes | enterprise RAG |

In [ ]:
from azure.ai.agents.models import AzureAISearchTool, AzureAISearchQueryType

search_conn = None
for conn in project.connections.list():
    if "search" in str(conn.type).lower():
        search_conn = conn
        break

if search_conn is None:
    print("No Azure AI Search connection on this project — skipping 6b.")
    print("Add one in the portal: Management center > Connected resources > + New connection.")
else:
    print("using connection:", search_conn.name, search_conn.id)
    try:
        ai_search = AzureAISearchTool(
            index_connection_id=search_conn.id,
            index_name=cfg.get("AZURE_SEARCH_INDEX", "ai103-index"),
            query_type=AzureAISearchQueryType.VECTOR_SEMANTIC_HYBRID,
            top_k=5,
        )
        searcher = agents.create_agent(
            model=cfg["MODEL_MINI"],
            name="ai103-search-agent",
            instructions="Answer from the search index only, and cite your sources.",
            tools=ai_search.definitions,
            tool_resources=ai_search.resources,
        )
        CREATED["agents"].append(searcher.id)

        t7 = agents.threads.create()
        CREATED["threads"].append(t7.id)
        agents.messages.create(thread_id=t7.id, role="user",
                               content="What does the returns policy say about opened items?")
        run7 = agents.runs.create_and_process(thread_id=t7.id, agent_id=searcher.id)
        print("status:", run7.status)
        print_thread(t7.id)
    except Exception as e:
        print("AzureAISearchTool failed:", type(e).__name__)
        print(str(e)[:350])
        print("\nUsual causes: the index does not exist (unit 02.2 deletes it), or the")
        print("Foundry managed identity lacks 'Search Index Data Reader'.")

## 7. OpenAPI spec tool and Bing grounding

Both are configured, not coded — and both are executed **by Foundry**, not by you.
The shapes below are what the exam expects you to recognise. They are not executed
here because each needs an external dependency (a reachable API; a billed Bing
connection).

In [ ]:
from azure.ai.agents.models import (
    OpenApiTool, OpenApiAnonymousAuthDetails, OpenApiConnectionAuthDetails,
    OpenApiConnectionSecurityScheme,
)

WEATHER_SPEC = {
    "openapi": "3.0.0",
    "info": {"title": "Weather", "version": "1.0.0"},
    "servers": [{"url": "https://api.example.com"}],
    "paths": {
        "/weather/{city}": {
            "get": {
                "operationId": "getWeather",
                "summary": "Current weather for a city",
                "parameters": [{
                    "name": "city", "in": "path", "required": True,
                    "schema": {"type": "string"},
                }],
                "responses": {"200": {"description": "OK"}},
            }
        }
    },
}

# Anonymous auth. For a secured API use OpenApiConnectionAuthDetails with a project
# connection holding the key, or managed identity — the credential never touches
# your code, which is a large part of the appeal.
openapi_tool = OpenApiTool(
    name="weather",
    spec=WEATHER_SPEC,
    description="Get current weather for a city. Use when the user asks about weather.",
    auth=OpenApiAnonymousAuthDetails(),
)

print("OpenApiTool definitions:")
for d in openapi_tool.definitions:
    print(" ", d.as_dict() if hasattr(d, "as_dict") else d)

print("\nFoundry calls this API itself. There is NO requires_action for an OpenAPI")
print("tool — your process is not involved in the HTTP call.")

In [ ]:
# Bing grounding. Requires a 'Grounding with Bing Search' resource connected to the
# project; it is billed separately from tokens. Results carry URL citations, and
# Microsoft's use terms require you to surface them.
from azure.ai.agents.models import BingGroundingTool

bing_conn = next((c for c in project.connections.list() if "bing" in str(c.type).lower()), None)

if bing_conn is None:
    print("No Grounding with Bing connection — showing the shape only:\n")
    print("    bing = BingGroundingTool(connection_id='<connection id>')")
    print("    agent = agents.create_agent(..., tools=bing.definitions)")
else:
    bing = BingGroundingTool(connection_id=bing_conn.id)
    grounded_agent = agents.create_agent(
        model=cfg["MODEL_MINI"],
        name="ai103-bing",
        instructions="Answer from web search results and always include the source URLs.",
        tools=bing.definitions,
    )
    CREATED["agents"].append(grounded_agent.id)
    t8 = agents.threads.create()
    CREATED["threads"].append(t8.id)
    agents.messages.create(thread_id=t8.id, role="user",
                           content="What is the latest GA version of the Azure AI Agents Python SDK?")
    agents.runs.create_and_process(thread_id=t8.id, agent_id=grounded_agent.id)
    print_thread(t8.id)

**Choosing between them, the way the exam frames it:**

| Requirement | Answer |
|---|---|
| Query an internal system that is not HTTP | Function calling |
| Approve or audit every call before it happens | Function calling (manual loop) |
| Call a documented REST API Foundry can reach | OpenAPI spec tool |
| Current public web information with citations | Grounding with Bing |
| Q&A over an index you already own | Azure AI Search tool |
| Q&A over a handful of PDFs, nothing built yet | File Search |
| Compute, chart, or transform data | Code Interpreter |
| Work that takes minutes and would exceed the run window | Azure Functions tool (async) |
| Extract structured fields from documents, images, audio, video | Content Understanding (unit 04.2), invoked as a function or via a pre-built index |

## 8. Run steps: the audit trail

`run_steps.list()` is where you find out what the agent actually did — which tools
fired, with what arguments, and what each step cost. This is the foundation of the
monitoring and error analysis in unit 02.4.

In [ ]:
def audit(thread_id, run_id):
    print(f"run {run_id}")
    total = 0
    for step in agents.run_steps.list(thread_id=thread_id, run_id=run_id,
                                      order=ListSortOrder.ASCENDING):
        usage = getattr(step, "usage", None)
        tokens = getattr(usage, "total_tokens", 0) or 0
        total += tokens
        print(f"  {step.type:<18} {step.status:<10} {tokens:>6} tokens")
        details = step.step_details
        for call in getattr(details, "tool_calls", []) or []:
            fn = getattr(call, "function", None)
            if fn is not None:
                print(f"      -> {fn.name}({fn.arguments})")
                out = (getattr(fn, 'output', '') or '')[:110]
                print(f"         returned: {out}")
            else:
                print(f"      -> {getattr(call, 'type', 'tool')}")
    print(f"  {'':<18} {'':<10} {total:>6} total")


audit(t2.id, r.id)

## 9. Cleanup — run this even if something above failed

Deleting an agent does **not** delete its threads. Threads, files, and vector stores
are independent objects that persist (and, for vector stores, bill) until removed.

In [ ]:
def safe(label, fn, ident):
    try:
        fn(ident)
        print(f"deleted {label:<14} {ident}")
    except Exception as e:
        print(f"failed  {label:<14} {ident}: {type(e).__name__}")


for vs_id in CREATED["vector_stores"]:
    safe("vector store", agents.vector_stores.delete, vs_id)
for file_id in CREATED["files"]:
    safe("file", agents.files.delete, file_id)
for thread_id in CREATED["threads"]:
    safe("thread", agents.threads.delete, thread_id)
for agent_id in CREATED["agents"]:
    safe("agent", agents.delete_agent, agent_id)

for path in [csv_path, policy_path]:
    try:
        os.remove(path)
    except (OSError, NameError):
        pass

print("\nAgents still on the project:")
remaining = [a for a in agents.list_agents()]
for a in remaining:
    print(f"  {a.id}  {a.name}")
if not remaining:
    print("  none")

## Exercise

Solutions at the bottom of [quiz.md](quiz.md).

1. **Starve the run.** Create an agent with `TOOL_SCHEMAS`, start a run with
   `runs.create`, and deliberately never submit tool outputs. Poll every 30 seconds
   and record the status transitions and how long until it changes. What is the
   terminal status, and what does `run.last_error` say?
2. **Description-driven routing.** Change `get_order_status`'s description to the
   uninformative string `Gets stuff.` and rerun the damaged-order conversation
   five times. How often is the tool called at all, and does it ever get called
   with a fabricated order ID?
3. **Enum enforcement.** Ask the agent to open a refund case with a reason not in
   the `enum` ("the colour is ugly"). What value arrives in your function, and what
   does that tell you about where enums are enforced compared with the strict JSON
   schema in unit 02.1?
4. **Thread growth.** Run a 10-turn conversation on one thread and plot
   `prompt_tokens` per turn. Fit a rough trend. Then repeat with a fresh thread each
   turn, injecting a one-line summary instead. Compare total cost and answer
   quality.

In [ ]:
# Your work here. Remember to delete anything you create.